In [26]:
import sqlite3
import pandas as pd
from sqlalchemy import create_engine

## 1. Connexion à la base SQLite
### 'example.db' sera créé si il n'existe pas.

In [27]:
conn = sqlite3.connect("example.db")
cur = conn.cursor()

## 2. Création de la table

In [28]:
cur.execute("DROP TABLE IF EXISTS actions")  # On supprime la table si elle existe
cur.execute("""
    CREATE TABLE actions (
        date TEXT,        -- Date de la transaction
        trans TEXT,       -- Type de transaction (BUY/SELL)
        action TEXT,      -- Nom de l'action
        quantite REAL,    -- Quantité d'actions
        prix REAL         -- Prix par action
    )
""")


### 3. Insertion de données

In [29]:
cur.execute("INSERT INTO actions VALUES ('2020-01-05','BUY','EPSILON',100,35.14)")
cur.execute("INSERT INTO actions VALUES ('2020-01-06','SELL','EPSILON',100,42)")

mouvements = [
    ('2020-03-28', 'BUY', 'IBM', 1000, 45.00),
    ('2020-04-05', 'BUY', 'MSFT', 1000, 72.00),
    ('2020-05-05', 'BUY', 'MSFT', 1000, 74.00),
    ('2020-06-05', 'BUY', 'MSFT', 1000, 61.00),
    ('2020-07-05', 'BUY', 'MSFT', 1000, 55.00),
    ('2020-04-06', 'SELL', 'IBM', 500, 53.00),
    ('2020-01-05','BUY','EPSILON',200,38),
    ('2020-02-05','SELL','EPSILON',300,35.14),
    ('2020-03-05','BUY','EPSILON',100,36),
    ('2020-04-05','BUY','EPSILON',500,37),
]
cur.executemany("INSERT INTO actions VALUES (?,?,?,?,?)", mouvements)
conn.commit()  # Toujours commit pour sauvegarder les modifications

### 4. Lecture avec pandas

In [ ]:
# On passe par sqlalchemy
engine = create_engine("sqlite:///example.db")

df = pd.read_sql_query("SELECT * FROM actions", engine)


Aperçu de toutes les transactions :
         date trans   action  quantite   prix
0  2020-01-05   BUY  EPSILON     100.0  35.14
1  2020-01-06  SELL  EPSILON     100.0  42.00
2  2020-03-28   BUY      IBM    1000.0  45.00
3  2020-04-05   BUY     MSFT    1000.0  72.00
4  2020-05-05   BUY     MSFT    1000.0  74.00


Aperçu de toutes les transactions :

In [ ]:
df.head()

### 5. Analyses simples avec pandas

#### Total d'actions achetées et vendues

In [ ]:
df.groupby("trans")["quantite"].sum()


Total d'actions par type de transaction :
trans
BUY     5900.0
SELL     900.0
Name: quantite, dtype: float64


#### Dépenses totales par action (pour les achats)


In [ ]:
# Les ordres d'achats
df_buy = df[df["trans"]=="BUY"]

In [ ]:
# Colonne intermédiaire pour calculer la dépense totale par action
df_buy["depense"] = df_buy["quantite"] * df_buy["prix"]

In [ ]:
# Total dépense par action
df_buy.groupby("action")["depense"].sum()


Total dépensé par action (BUY) :
action
EPSILON     33214.0
IBM         45000.0
MSFT       262000.0
Name: depense, dtype: float64


#### Prix moyen de chaque action achetée

In [ ]:
df_buy.groupby("action")["prix"].mean()


Prix moyen par action achetée :
action
EPSILON    36.535
IBM        45.000
MSFT       65.500
Name: prix, dtype: float64


#### Transaction la plus chère

In [ ]:
df.loc[df["prix"].idxmax()]


Transaction la plus chère :
date        2020-05-05
trans              BUY
action            MSFT
quantite        1000.0
prix              74.0
Name: 4, dtype: object


#### Transaction la moins chère

In [ ]:
df.loc[df["prix"].idxmin()]



Transaction la moins chère :
date        2020-01-05
trans              BUY
action         EPSILON
quantite         100.0
prix             35.14
Name: 0, dtype: object


### 7. Traitement par morceau

In [ ]:
import random

from pathlib import Path

db_path = "example.db"
table_name = "transactions_massives"
rows_per_batch = 100_000
n_batches = 10

conn_mass = sqlite3.connect(db_path)
cur_mass = conn_mass.cursor()

cur_mass.execute(f"DROP TABLE IF EXISTS {table_name}")
cur_mass.execute(f"""
    CREATE TABLE {table_name} (
        date TEXT,
        trans TEXT,
        action TEXT,
        quantite REAL,
        prix REAL
    )
""")
conn_mass.commit()

actions = ["EPSILON", "IBM", "MSFT", "AAPL", "NVDA"]
types_trans = ["BUY", "SELL"]
rng = random.Random(42)

for batch_idx in range(n_batches):
    batch = []

    for _ in range(rows_per_batch):
        month = rng.randint(1, 12)
        day = rng.randint(1, 28)
        date = f"2024-{month:02d}-{day:02d}"
        trans = rng.choice(types_trans)
        action = rng.choice(actions)
        quantite = float(rng.randint(100, 999))
        prix = round(rng.uniform(20, 95), 2)
        batch.append((date, trans, action, quantite, prix))

    cur_mass.executemany(
        f"INSERT INTO {table_name} VALUES (?, ?, ?, ?, ?)",
        batch,
    )
    conn_mass.commit()

    total_rows = (batch_idx + 1) * rows_per_batch
    size_mb = Path(db_path).stat().st_size / (1024 * 1024)
    print(f"Batch {batch_idx + 1}/{n_batches} | lignes: {total_rows:,} | taille DB: {size_mb:.2f} Mo")

cur_mass.close()
conn_mass.close()

In [ ]:
chunksize = 50_000
sum_prix = 0.0
count_prix = 0

for chunk in pd.read_sql_query(
    f"SELECT prix FROM {table_name}",
    engine,
    chunksize=chunksize,
):
    sum_prix += chunk["prix"].sum()
    count_prix += len(chunk)

prix_moyen = sum_prix / count_prix
print(f"Lignes traitées : {count_prix:,}")
print(f"Prix moyen : {prix_moyen:.4f}")

### 8. Fermeture de la connexion

In [37]:
conn.close()